# AICE 머신러닝 연습 — 회귀 17문제

시험 샘플과 비슷한 기본 작업 흐름으로 만든 연습 문제입니다. 실제 출제나 출제 범위를 보장하는 자료는 아닙니다. **딥러닝은 제외하고 머신러닝만 연습합니다.**

- CSV: `aice_regression_1122.csv`를 노트북과 같은 폴더에 두세요.
- 처음부터 차례로 실행합니다. 답안 셀은 직접 작성하며 정답 파일은 별도입니다.
- **2분할만 사용합니다.** X_train, X_valid, y_train, y_valid 이름을 일관되게 사용합니다.
- Pipeline, 사용자 정의 함수, 3분할, PCA, 군집화는 사용하지 않습니다.
- 새로 보는 도구는 각 문제의 '사용 이유'와 힌트부터 확인하세요. 모든 모델 코드는 직접 fit → predict → 평가 형태입니다.
- 정답 코드 실행 확인 환경: scikit-learn 1.8.0. 설치: `pip install numpy pandas matplotlib scikit-learn joblib ipykernel`.

## 데이터 안내
실제 고객 정보가 아닌 합성 데이터입니다. score 열은 모델 연습용 수치이며 현실의 금액·개월을 뜻하지 않습니다. ID는 제거하고, 정답은 monthly_spend(합성 월 지출액)입니다. 입력에는 결측치가 있습니다.

## 전처리 안내
이번 연습은 시험식 단계별 조작을 익히기 위해 결측치 대체·인코딩을 먼저 수행합니다. 실제 성능을 엄격히 평가할 때는 분할 후 학습 데이터에서만 중앙값·최빈값·인코딩 기준을 구해 검증 데이터에 적용해야 합니다. StandardScaler는 이번에도 학습 데이터에만 fit합니다. 검증 점수로 여러 번 모델을 고르므로 이 점수를 독립적인 최종 성능이라고 해석하지 않습니다.


## 01. 데이터 불러오기

**사용 이유:** read_csv는 CSV를 데이터프레임으로 읽습니다.

**문제:** CSV를 읽어 data에 저장하고 앞 5행을 출력하세요.

<details><summary>힌트 보기</summary>

pd.read_csv(), head()

</details>

In [37]:
# 여기에 답안코드를 작성하세요.
import pandas as pd

data = pd.read_csv("aice_regression_1122.csv")

data.head(5)

,customer_id,visits_score,basket_score,discount_score,tenure_score,delivery_score,promotion_score,segment,region,monthly_spend
0,C00000,1.411690,-0.238165,0.138622,-2.658049,-0.104518,0.860201,medium,seoul,187319.0
1,C00001,-0.159812,-0.722423,1.251315,0.712237,-1.303260,-0.670359,small,seoul,154685.0
2,C00002,0.201219,0.072085,1.032157,0.033732,0.450408,-0.718242,medium,other,209859.0
3,C00003,-1.494931,-0.323858,0.246020,0.544869,0.468091,-0.758721,medium,other,192775.0
4,C00004,-0.761850,-0.976698,-0.577122,1.149418,-0.824751,-0.524227,small,busan,126647.0


## 02. 데이터 구조 확인

**사용 이유:** shape는 행·열 수, info는 자료형과 결측 여부, describe는 수치형 요약을 보여줍니다.

**문제:** 행·열 수, 열별 자료형, 수치형 기술통계를 확인하세요.

<details><summary>힌트 보기</summary>

shape, info(), describe()

</details>

In [38]:
# 여기에 답안코드를 작성하세요.
# 행 열 수
print(data.shape)
# 열별 자료형
print(data.info())
# 수치형 기술 통계 
data.describe()

(1500, 10)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      1500 non-null   str    
 1   visits_score     1460 non-null   float64
 2   basket_score     1460 non-null   float64
 3   discount_score   1460 non-null   float64
 4   tenure_score     1460 non-null   float64
 5   delivery_score   1460 non-null   float64
 6   promotion_score  1460 non-null   float64
 7   segment          1460 non-null   str    
 8   region           1460 non-null   str    
 9   monthly_spend    1500 non-null   float64
dtypes: float64(7), str(3)
memory usage: 117.3 KB
None


,visits_score,basket_score,discount_score,tenure_score,delivery_score,promotion_score,monthly_spend
count,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1500.000000
mean,0.008837,-0.043361,0.030254,-0.047636,0.003383,-0.014833,203462.325333
std,0.997912,0.980553,0.980651,1.005933,1.008217,1.013891,73508.201636
min,-3.699500,-3.798277,-3.176029,-3.464063,-3.139150,-3.264902,5000.000000
25%,-0.681238,-0.677746,-0.620785,-0.698367,-0.671793,-0.706320,154673.750000
50%,0.006105,-0.048736,0.043343,-0.056418,-0.019084,-0.016491,197459.500000
75%,0.697669,0.618822,0.707079,0.656814,0.688118,0.679314,242181.750000
max,3.369276,2.603986,3.105631,3.730192,3.110532,3.306965,575838.000000


## 03. 결측치 확인

**사용 이유:** isna().sum()은 각 열의 결측치 수를 셉니다.

**문제:** 열별 결측치 수를 출력하세요.

<details><summary>힌트 보기</summary>

data.isna().sum()

</details>

In [39]:
# 여기에 답안코드를 작성하세요.
print(data.isna().sum())

customer_id         0
visits_score       40
basket_score       40
discount_score     40
tenure_score       40
delivery_score     40
promotion_score    40
segment            40
region             40
monthly_spend       0
dtype: int64


## 04. 결측치 대체

**사용 이유:** fillna는 누락값을 채웁니다. 중앙값은 수치형, 최빈값은 범주형 대체에 사용할 수 있습니다.

**문제:** 수치형 입력 열은 각 열의 중앙값, segment/region은 각 열의 최빈값으로 대체하세요.

<details><summary>힌트 보기</summary>

median(), mode()[0], fillna()

</details>

In [40]:
# 여기에 답안코드를 작성하세요.
data.head()


,customer_id,visits_score,basket_score,discount_score,tenure_score,delivery_score,promotion_score,segment,region,monthly_spend
0,C00000,1.411690,-0.238165,0.138622,-2.658049,-0.104518,0.860201,medium,seoul,187319.0
1,C00001,-0.159812,-0.722423,1.251315,0.712237,-1.303260,-0.670359,small,seoul,154685.0
2,C00002,0.201219,0.072085,1.032157,0.033732,0.450408,-0.718242,medium,other,209859.0
3,C00003,-1.494931,-0.323858,0.246020,0.544869,0.468091,-0.758721,medium,other,192775.0
4,C00004,-0.761850,-0.976698,-0.577122,1.149418,-0.824751,-0.524227,small,busan,126647.0


In [41]:
num_cols = ["visits_score", "basket_score", "discount_score", "tenure_score", "delivery_score", "promotion_score"]
str_cols = ["segment", "region"]

for i in num_cols:
    data[i] = data[i].fillna(data[i].median())

for i in str_cols:
    data[i] = data[i].fillna(data[i].mode().iloc[0])

data.isna().sum()

customer_id        0
visits_score       0
basket_score       0
discount_score     0
tenure_score       0
delivery_score     0
promotion_score    0
segment            0
region             0
monthly_spend      0
dtype: int64

## 05. 불필요한 열 제거

**사용 이유:** 식별자는 고객을 구별하는 값으로, 이번 예측의 입력에서 제외합니다.

**문제:** customer_id 열을 제거하고 data에 다시 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[...])

</details>

In [42]:
# 여기에 답안코드를 작성하세요.
data = data.drop(columns=["customer_id"])
data

,visits_score,basket_score,discount_score,tenure_score,delivery_score,promotion_score,segment,region,monthly_spend
0,1.411690,-0.238165,0.138622,-2.658049,-0.104518,0.860201,medium,seoul,187319.0
1,-0.159812,-0.722423,1.251315,0.712237,-1.303260,-0.670359,small,seoul,154685.0
2,0.201219,0.072085,1.032157,0.033732,0.450408,-0.718242,medium,other,209859.0
3,-1.494931,-0.323858,0.246020,0.544869,0.468091,-0.758721,medium,other,192775.0
4,-0.761850,-0.976698,-0.577122,1.149418,-0.824751,-0.524227,small,busan,126647.0
...,...,...,...,...,...,...,...,...,...
1495,0.569315,-0.480078,0.634208,0.141676,0.100294,0.572145,medium,busan,224283.0
1496,0.497229,0.828605,-1.145448,-0.487016,-0.590572,0.753418,large,other,196043.0
1497,0.402056,0.566328,-1.355439,0.959389,-1.867426,-1.205186,medium,other,184372.0
1498,-0.848728,0.016700,-0.858159,-0.576021,-0.550847,0.947065,medium,other,116837.0


## 06. 원핫 인코딩

**사용 이유:** get_dummies는 범주형 값을 0/1 열로 바꿉니다.

**문제:** 범주형 열을 원핫 인코딩하여 data_encoded에 저장하세요. dtype=int를 지정하세요.

<details><summary>힌트 보기</summary>

pd.get_dummies(data, columns=[...], dtype=int)

</details>

In [43]:
# 여기에 답안코드를 작성하세요.
data_encoded = pd.get_dummies(
    data,
    columns=["segment", "region"],
    dtype=int
)

data_encoded

,visits_score,basket_score,discount_score,tenure_score,delivery_score,promotion_score,monthly_spend,segment_large,segment_medium,segment_small,region_busan,region_other,region_seoul
0,1.411690,-0.238165,0.138622,-2.658049,-0.104518,0.860201,187319.0,0,1,0,0,0,1
1,-0.159812,-0.722423,1.251315,0.712237,-1.303260,-0.670359,154685.0,0,0,1,0,0,1
2,0.201219,0.072085,1.032157,0.033732,0.450408,-0.718242,209859.0,0,1,0,0,1,0
3,-1.494931,-0.323858,0.246020,0.544869,0.468091,-0.758721,192775.0,0,1,0,0,1,0
4,-0.761850,-0.976698,-0.577122,1.149418,-0.824751,-0.524227,126647.0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,0.569315,-0.480078,0.634208,0.141676,0.100294,0.572145,224283.0,0,1,0,1,0,0
1496,0.497229,0.828605,-1.145448,-0.487016,-0.590572,0.753418,196043.0,1,0,0,0,1,0
1497,0.402056,0.566328,-1.355439,0.959389,-1.867426,-1.205186,184372.0,0,1,0,0,1,0
1498,-0.848728,0.016700,-0.858159,-0.576021,-0.550847,0.947065,116837.0,0,1,0,0,1,0


## 07. 입력과 정답 분리

**사용 이유:** X는 예측에 사용할 입력, y는 맞혀야 하는 정답입니다.

**문제:** monthly_spend을 y, 나머지 열을 X에 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[정답열])로 X를 만듭니다.

</details>

In [44]:
# 여기에 답안코드를 작성하세요.
X = data_encoded.drop(columns=["monthly_spend"])
y = data_encoded["monthly_spend"]

## 08. 학습·검증 2분할

**사용 이유:** train_test_split은 데이터를 두 묶음으로 나눕니다. 연속형 정답에는 stratify=y를 지정하지 않습니다.

**문제:** 학습 80%, 검증 20%로 분할하세요. 변수는 X_train, X_valid, y_train, y_valid, random_state=42를 사용하세요.

<details><summary>힌트 보기</summary>

train_test_split(X, y, ...)의 반환 순서를 확인하세요.

</details>

In [45]:
# 여기에 답안코드를 작성하세요.
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    # 연속형 정답에는 stratify=y 지정 안함
)

## 09. 표준화

**사용 이유:** StandardScaler는 학습 데이터의 평균·표준편차를 기준으로 값을 변환합니다. 거리·계수 기반 모델에서 유용합니다.

**문제:** StandardScaler를 scaler로 만들고, 학습 데이터는 fit_transform하여 X_train_scaled, 검증 데이터는 transform하여 X_valid_scaled에 저장하세요.

<details><summary>힌트 보기</summary>

검증 데이터에는 fit_transform을 사용하지 않습니다.

</details>

In [46]:
# 여기에 답안코드를 작성하세요.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

## 10. 선형 회귀 학습

**사용 이유:** LinearRegression은 입력의 선형 조합으로 연속 수치를 예측합니다.

**문제:** LinearRegression을 lr로 만들고 표준화한 학습 입력으로 학습하세요. 검증 예측은 pred_lr에 저장하세요.

<details><summary>힌트 보기</summary>

회귀 예측은 클래스 번호가 아닌 수치입니다.

</details>

In [47]:
# 여기에 답안코드를 작성하세요.
from sklearn.linear_model import LinearRegression  

lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict(X_valid_scaled)

## 11. 회귀 평가지표

**사용 이유:** MAE는 평균 절대오차, MSE는 평균 제곱오차, RMSE는 MSE의 제곱근입니다. 셋 모두 작을수록 좋습니다. R²는 클수록 좋으며 음수도 가능합니다.

**문제:** lr의 검증 MAE, MSE, RMSE, R²를 각각 출력하세요.

<details><summary>힌트 보기</summary>

RMSE=np.sqrt(mean_squared_error(...)). 회귀에 accuracy_score를 사용하지 않습니다.

</details>

In [48]:
# 여기에 답안코드를 작성하세요.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 실제 지출액과 예측값이 평균적으로 이정도 차이
mae = mean_absolute_error(y_valid, pred_lr)
# 오차를 제곱해서 평균한 값이라 숫자가 큼
mse = mean_squared_error(y_valid, pred_lr)
# 큰 오차에 더 민감한 지표. 다시 원 단위로 돌아옴
rmse = np.sqrt(mse)
# 검증 정답의 변동을 설명
r2 = r2_score(y_valid, pred_lr)

met = [mae, mse, rmse, r2]

for i in met:
    print(i)

26748.95091393904
1944086826.348757
44091.79998989333
0.6538063771494109


## 12. 회귀 의사결정나무

**사용 이유:** DecisionTreeRegressor는 분기된 구간별 수치를 예측합니다. 분류용 Classifier와 이름을 구분하세요.

**문제:** max_depth=5, min_samples_split=3, random_state=42로 dt를 만들고 원본 입력으로 학습하세요. pred_dt를 만들고 MAE를 출력하세요.

<details><summary>힌트 보기</summary>

DecisionTreeRegressor를 import합니다.

</details>

In [49]:
# 여기에 답안코드를 작성하세요.
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=5, min_samples_split=3, random_state=42)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_valid)

print(mean_absolute_error(y_valid, pred_dt))

38717.60564858719


## 13. 회귀 랜덤 포레스트

**사용 이유:** RandomForestRegressor는 여러 회귀 트리의 예측값을 평균합니다.

**문제:** n_estimators=100, max_depth=8, random_state=42로 rf를 만들고 원본 입력으로 학습하세요. pred_rf를 만들고 MAE와 RMSE를 출력하세요.

<details><summary>힌트 보기</summary>

회귀에는 predict_proba가 아닌 predict를 사용합니다.

</details>

In [50]:
# 여기에 답안코드를 작성하세요.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_valid)

print(mean_absolute_error(y_valid, pred_rf))
print(np.sqrt(mean_squared_error(y_valid, pred_rf)))

24999.187732028553
36207.74410204219


## 14. 회귀 모델 비교

**사용 이유:** 동일 검증 데이터와 동일 지표로 비교해야 합니다. MAE는 정답과 같은 단위입니다.

**문제:** lr, dt, rf의 검증 MAE를 각각 출력하고 가장 작은 모델을 확인하세요.

<details><summary>힌트 보기</summary>

이번 정답의 단위는 합성 지출액 원입니다.

</details>

In [51]:
# 여기에 답안코드를 작성하세요.
print(f"lr mae : {mean_absolute_error(y_valid, pred_lr)}")
print(f"dt mae : {mean_absolute_error(y_valid, pred_dt)}")
print(f"rf mae : {mean_absolute_error(y_valid, pred_rf)}")

lr mae : 26748.95091393904
dt mae : 38717.60564858719
rf mae : 24999.187732028553


## 15. 회귀 GridSearchCV

**사용 이유:** scikit-learn 탐색은 점수가 클수록 좋다고 처리하므로 MAE에 음수를 붙인 neg_mean_absolute_error를 사용합니다.

**문제:** RandomForestRegressor(n_estimators=100, random_state=42)에 max_depth=[5,10], min_samples_split=[2,5]를 탐색하세요. cv=3, scoring="neg_mean_absolute_error", 변수는 gs_rf입니다.

<details><summary>힌트 보기</summary>

best_score_의 부호를 바꾸면 MAE입니다.

</details>

In [52]:
# 여기에 답안코드를 작성하세요.
from sklearn.model_selection import GridSearchCV

params = {
    "max_depth":[5,10],
    "min_samples_split":[2,5]
}

gs_rf = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42),
    params,
    cv=3,
    scoring="neg_mean_absolute_error"
)

gs_rf.fit(X_train, y_train);

## 16. 최적 회귀 모델 검증

**사용 이유:** best_estimator_로 최적 모델을 꺼내고 별도 검증 데이터에서 평가합니다.

**문제:** 최적 모델을 best_model, 검증 예측을 best_pred에 저장하고 MAE, RMSE, R²를 출력하세요.

<details><summary>힌트 보기</summary>

gs_rf.best_score_와 검증 점수는 서로 다른 데이터에서 계산됩니다.

</details>

In [53]:
# 여기에 답안코드를 작성하세요.
best_model = gs_rf.best_estimator_
best_pred = gs_rf.predict(X_valid)

print(mean_absolute_error(y_valid, best_pred))
print(np.sqrt(mean_squared_error(y_valid, best_pred)))
print(r2_score(y_valid, best_pred))

23141.84223317654
34403.51009940743
0.7892299416766685


## 17. 회귀 모델 저장·불러오기

**사용 이유:** joblib으로 저장한 모델은 동일한 전처리와 열 순서의 입력을 받아야 합니다.

**문제:** best_model을 aice_best_regressor.joblib로 저장하고 loaded_model로 불러와 검증 앞 5행을 예측하세요.

<details><summary>힌트 보기</summary>

모델을 저장해도 결측치 대체와 원핫 인코딩 과정이 자동으로 포함되지는 않습니다.

</details>

In [54]:
# 여기에 답안코드를 작성하세요.
import joblib
joblib.dump(best_model, 'aice_best_regressor.joblib')
loaded_model = joblib.load("aice_best_regressor.joblib")
print(loaded_model.predict(X_valid.iloc[:5]))

[141094.44650079 221714.77237185 219727.09300269 153307.56224876
 167565.21129482]
